# 🤖 AI agents with CrewAI (Groq + live Google search)

A normal chatbot answers **once**, from memory. An **agent** can **use tools**, look at the results, and **decide what to do next**, again and again, until the job is done.

**Agent = LLM 🧠 + tools 🛠️ + a loop 🔁**. In the HTML pages we saw the loop: **Think → Act → Observe → repeat**.
Here we don't write that loop ourselves: **CrewAI** (a free, open-source agent framework) runs it for us.

| Piece | In this notebook |
|---|---|
| 🧠 LLM (the brain that decides) | open-source **Llama 3.3** on **Groq** |
| 🛠️ Tools (what the agent can do) | **Google Search** (live, through SerpAPI) and a **calculator** |
| 🔁 The loop | **CrewAI** |
| 👥 A crew | several agents, each with its own job, working one after another |

> Matching HTML pages: `RAG_Agents_Visualized/05_what_is_an_agent.html` and `06_agent_step_by_step.html`.
> 🔑 You need 2 free keys (Groq and SerpAPI). See `00_setup.html`.

## 0. Install the libraries (run once)

Same exact versions as `requirements.txt`. On Colab, if it asks you to **restart the session**, do it and continue from the next cell.

In [ ]:
%pip install -q "crewai[litellm]==1.15.23" litellm==1.104.0 crewai-tools==1.15.23 serpapi==1.1.2

## 0. Load your API key(s)

The cell looks for each key in this order:
1. **Colab:** the 🔑 **Secrets** panel on the left (add the key there once, with notebook access switched on).
2. **Laptop:** an environment variable (see `RAG_Agents_Visualized/00_setup.html`).
3. Otherwise it **asks you** to paste it.

**Never type a key directly into the code.** It would end up on GitHub, and anyone could use it.

In [ ]:
import os
from getpass import getpass

def load_key(name):
    if os.environ.get(name):                      # laptop: environment variable
        return
    try:
        from google.colab import userdata         # Colab: 🔑 Secrets panel
        os.environ[name] = userdata.get(name)
    except Exception:
        os.environ[name] = getpass(f"Paste your {name}: ")   # otherwise: ask

load_key("GROQ_API_KEY")
load_key("SERPAPI_API_KEY")
print("Keys loaded ✅")

os.environ["CREWAI_TRACING_ENABLED"] = "false"    # no cloud tracing prompt from CrewAI

## 1. The brain: Llama 3.3 on Groq

One line. The `groq/` prefix tells CrewAI which company runs the model.

In [ ]:
from crewai import LLM

llm = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)   # model list: console.groq.com/docs/models

print(llm.call("In one sentence: what is an AI agent?"))

## 2. The tools

A CrewAI tool is just a **Python function** with the `@tool` decorator.
The **docstring** (the text in triple quotes) is important: the LLM reads it to decide **when** to use the tool.

**Tool 1: Google Search.** We use SerpAPI's official `serpapi` library and keep only the top 5 results.
*(CrewAI also has a ready-made `SerpApiGoogleSearchTool`, but it returns the whole Google page, thousands of words, which uses up Groq's free limit very fast.)*

In [ ]:
import serpapi
from crewai.tools import tool

@tool("Google Search")
def google_search(query: str) -> str:
    """Search Google for up-to-date facts, news, dates and numbers. Input: what to search for."""
    results = serpapi.Client(api_key=os.environ["SERPAPI_API_KEY"]).search({"engine": "google", "q": query})

    lines = []
    box = results.get("answer_box", {})                       # Google's quick answer, if any
    if box.get("answer") or box.get("snippet"):
        lines.append("Quick answer: " + str(box.get("answer") or box.get("snippet")))
    for r in results.get("organic_results", [])[:5]:          # the top 5 normal results
        lines.append(f"- {r.get('title')}: {r.get('snippet')} ({r.get('link')})")
    return "\n".join(lines) or "No results found."

print(google_search.run(query="Eiffel Tower completed year"))

**Tool 2: Calculator.** LLMs often get exact math wrong, so we give the agent a calculator.
It only accepts numbers and `+ - * / ( )`, so the LLM can't make it run any other Python code.

In [ ]:
@tool("Calculator")
def calculator(expression: str) -> str:
    """Calculate a math expression exactly, for example '2010 - 1889' or '(18 + 4) * 2'."""
    if not set(expression) <= set("0123456789+-*/(). ") or "**" in expression:
        return "Error: only numbers and + - * / ( ) are allowed"
    return str(eval(expression))      # safe: only numbers and math symbols can reach this line

print(calculator.run(expression="2010 - 1889"))

## 3. One agent, one task

In CrewAI you describe an agent like a **job post**:
- **role**: the job title
- **goal**: what it tries to achieve
- **backstory**: its personality and experience (this shapes how it works)

A **Task** is the work you give it, plus what the result should look like.
`verbose=True` prints the agent's **Thought → Action → Observation** loop, just like page 06.

In [ ]:
from crewai import Agent, Task, Crew

researcher = Agent(
    role="Research Assistant",
    goal="Find correct, up-to-date facts using Google, and do any math with the calculator",
    backstory="You are careful. You never guess: you search first and check your numbers.",
    tools=[google_search, calculator],
    llm=llm,
    max_iter=5,          # safety: at most 5 rounds of the loop
    verbose=True,
)

question_task = Task(
    description="Answer this question: {question}",
    expected_output="A short answer (2-3 sentences) that mentions the facts you found.",
    agent=researcher,
)

crew = Crew(agents=[researcher], tasks=[question_task], verbose=True)
result = crew.kickoff(inputs={"question": "How old was the Eiffel Tower when the Burj Khalifa opened?"})

print("\n✅ FINAL ANSWER:\n", result.raw)

👀 **Notice in the log above:** the agent searched **twice** (once for each building), then used the **calculator**, then answered.
Nobody told it those steps. The LLM **decided** them, and CrewAI **ran** the tools.

## 4. ✨ A crew: two agents working as a team

Real work is split between people. Here:
1. 🔎 **Researcher** searches Google for the latest news on a topic.
2. ✍️ **Writer** turns those notes into a short LinkedIn post. (No tools: it only writes.)

CrewAI runs the tasks **in order** and passes the researcher's notes to the writer automatically.

In [ ]:
from crewai import Process
from IPython.display import Markdown, display

news_researcher = Agent(
    role="Tech News Researcher",
    goal="Find the 3 most important recent developments about {topic}",
    backstory="You are a tech journalist who checks every fact on Google and always notes the source.",
    tools=[google_search],
    llm=llm,
    max_iter=5,
    verbose=True,
)

writer = Agent(
    role="LinkedIn Writer",
    goal="Turn research notes into a short, clear LinkedIn post about {topic}",
    backstory="You write simply, for students. No hype, no jargon.",
    llm=llm,
    verbose=True,
)

research_task = Task(
    description="Search for the latest news about {topic}. Pick the 3 most important points.",
    expected_output="3 bullet points, each with one sentence and its source link.",
    agent=news_researcher,
)

write_task = Task(
    description="Write a LinkedIn post about {topic} using the research notes.",
    expected_output="A post of at most 150 words in markdown, with a catchy first line and 3 hashtags.",
    agent=writer,
)

crew = Crew(agents=[news_researcher, writer], tasks=[research_task, write_task],
            process=Process.sequential, verbose=True)

result = crew.kickoff(inputs={"topic": "AI in healthcare in India"})
display(Markdown(result.raw))

Now try your own topic:

In [ ]:
result = crew.kickoff(inputs={"topic": "electric vehicles in India"})
display(Markdown(result.raw))

---
## 🧠 Recap

- **Agent** = LLM + tools + loop. **CrewAI** gives us the loop, so we only describe agents, tools and tasks.
- A **tool** is a Python function with `@tool` and a clear docstring.
- A **crew** = several agents with different jobs, passing their work along like a team.

## 🛠️ Make it your own (your project)

Change the **agents**, **tasks** and **tools**. The rest stays the same.

| Project idea | Agents | Tools |
|---|---|---|
| 📄 **PDF + web agent**: "Is my company's leave better than the Indian average?" | Researcher | Google Search + a `search_pdf` tool (the `search` function from notebook 01) |
| 🌦️ **Trip planner**: "Plan 2 days in Goa this weekend" | Weather checker → Planner | Google Search |
| 🎓 **Career coach**: "What skills do ML jobs in Bangalore ask for?" | Job researcher → Coach | Google Search |
| 💹 **Stock explainer**: "Why did this stock move today?" | News researcher → Analyst | Google Search + Calculator |
| 📰 **Daily news digest** | Researcher → Writer → Editor | Google Search |

**Experiments to try:**
1. Set `max_iter=1` on the researcher. What happens to the Eiffel Tower question?
2. Remove the `calculator` tool. Does the agent still get the math right?
3. Change the writer's `backstory` to *"You write like a pirate"*. 🏴‍☠️

**To put it on your resume:**
- Keys only in Colab Secrets or environment variables. Never push them to GitHub.
- Add a README with an example run (the Thought → Action → Observation log is great to show).
- Resume line: *Built a multi-agent AI system with CrewAI that searches the web live (Python, Groq API, Llama 3.3, SerpAPI).*